# Ziffernerkennung – Handschrift mit dem Computer lesen

In diesem Notebook bauen wir ein System, das **handgeschriebene Ziffern** (0–9) erkennt.

Der Plan:
1. **Zeichenfeld**: Wir bauen ein interaktives Zeichenfeld, in dem du mit der Maus eine Ziffer malen kannst
2. **Vorverarbeitung**: Die Zeichnung wird in ein 28x28-Pixel-Bild umgewandelt (das Format des MNIST-Datensatzes)
3. **Klassifikation** (kommt später): Ein trainiertes Modell erkennt, welche Ziffer du gemalt hast

---

## Teil 1: Das Zeichenfeld

Wir nutzen **matplotlib** im interaktiven Modus (`%matplotlib widget`), um ein Zeichenfeld zu bauen. Du kannst darin mit der Maus eine Ziffer malen.

Das Zeichenfeld ist 280x280 Pixel groß – das ist 10x so groß wie die MNIST-Bilder (28x28), damit man bequem zeichnen kann. Beim Übernehmen wird das Bild automatisch herunterskaliert.

In [ ]:
%matplotlib widget
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.widgets import Button as MplButton
from PIL import Image
import io

In [ ]:
# ── Zeichenfeld ──

# Globale Variable: hier landet das fertige 28x28-Bild
ziffer_bild = None

# 280x280 Pixel-Leinwand (weiß)
leinwand = np.ones((280, 280), dtype=np.uint8) * 255
stiftbreite = 8  # Radius in Pixeln

# --- Figure aufbauen ---
fig = plt.figure(figsize=(5, 5.8))

# Zeichenfläche (oben)
ax_draw = fig.add_axes([0.1, 0.18, 0.8, 0.75])
bild_anzeige = ax_draw.imshow(leinwand, cmap="gray", vmin=0, vmax=255, interpolation="nearest")
ax_draw.set_title("Male eine Ziffer (0–9) mit der Maus")
ax_draw.set_xticks([])
ax_draw.set_yticks([])

# Buttons (unten)
ax_btn_clear = fig.add_axes([0.15, 0.04, 0.3, 0.06])
ax_btn_submit = fig.add_axes([0.55, 0.04, 0.3, 0.06])
btn_clear = MplButton(ax_btn_clear, "Löschen")
btn_submit = MplButton(ax_btn_submit, "Übernehmen")

# --- Zeichen-Logik ---
zeichnet = False
letzte_pos = None

def strich_malen(x, y):
    """Malt einen ausgefüllten Kreis an Position (x, y)."""
    yy, xx = np.ogrid[:280, :280]
    maske = (xx - x)**2 + (yy - y)**2 <= stiftbreite**2
    leinwand[maske] = 0  # Schwarz

def linie_malen(x0, y0, x1, y1):
    """Malt eine Linie von (x0,y0) nach (x1,y1) durch viele kleine Kreise."""
    dist = max(abs(x1 - x0), abs(y1 - y0), 1)
    for t in np.linspace(0, 1, int(dist) + 1):
        x = int(x0 + t * (x1 - x0))
        y = int(y0 + t * (y1 - y0))
        strich_malen(x, y)

def on_press(event):
    global zeichnet, letzte_pos
    if event.inaxes != ax_draw:
        return
    zeichnet = True
    x, y = int(event.xdata), int(event.ydata)
    letzte_pos = (x, y)
    strich_malen(x, y)
    bild_anzeige.set_data(leinwand)
    fig.canvas.draw_idle()

def on_move(event):
    global letzte_pos
    if not zeichnet or event.inaxes != ax_draw:
        return
    x, y = int(event.xdata), int(event.ydata)
    if letzte_pos is not None:
        linie_malen(letzte_pos[0], letzte_pos[1], x, y)
    letzte_pos = (x, y)
    bild_anzeige.set_data(leinwand)
    fig.canvas.draw_idle()

def on_release(event):
    global zeichnet, letzte_pos
    zeichnet = False
    letzte_pos = None

def on_clear(event):
    leinwand[:] = 255
    bild_anzeige.set_data(leinwand)
    fig.canvas.draw_idle()

def on_submit(event):
    global ziffer_bild
    # PIL-Bild erstellen und auf 28x28 herunterskalieren
    bild = Image.fromarray(leinwand, mode="L")
    bild_28 = bild.resize((28, 28), Image.LANCZOS)

    # Invertieren: MNIST hat weiße Schrift auf schwarzem Grund
    ziffer_bild = 255 - np.array(bild_28)
    print(f"\nBild übernommen! Form: {ziffer_bild.shape}, Werte: {ziffer_bild.min()}–{ziffer_bild.max()}")

# Events verbinden
fig.canvas.mpl_connect("button_press_event", on_press)
fig.canvas.mpl_connect("motion_notify_event", on_move)
fig.canvas.mpl_connect("button_release_event", on_release)
btn_clear.on_clicked(on_clear)
btn_submit.on_clicked(on_submit)

plt.show()

---

## Teil 2: Vorschau des 28x28-Bildes

Nachdem du eine Ziffer gemalt und auf **Übernehmen** geklickt hast, kannst du dir das herunterskalierte 28x28-Bild anschauen. So sieht es aus, wenn es an einen MNIST-Classifier übergeben wird.

In [ ]:
# Vorschau: So sieht das Bild im MNIST-Format aus

if ziffer_bild is None:
    print("Noch kein Bild übernommen! Male oben eine Ziffer und klicke 'Übernehmen'.")
else:
    fig2, (ax1, ax2) = plt.subplots(1, 2, figsize=(7, 3.5))

    ax1.imshow(ziffer_bild, cmap="gray", vmin=0, vmax=255)
    ax1.set_title("28x28 Pixel (MNIST-Format)")
    ax1.axis("off")

    ax2.imshow(ziffer_bild, cmap="gray", vmin=0, vmax=255, interpolation="nearest")
    ax2.set_title("Pixel-Ansicht (vergrößert)")
    ax2.set_xticks(range(0, 28, 7))
    ax2.set_yticks(range(0, 28, 7))
    ax2.grid(True, color="gray", linewidth=0.3)

    fig2.tight_layout()
    plt.show()

    print(f"Bildgröße:    {ziffer_bild.shape}")
    print(f"Wertebereich: {ziffer_bild.min()} – {ziffer_bild.max()}")
    anteil = (ziffer_bild > 20).sum() / ziffer_bild.size * 100
    print(f"Aktive Pixel:  {anteil:.1f}%")
    print(f"\nDas Bild liegt in der Variable 'ziffer_bild' als NumPy-Array bereit.")

---

## Nächster Schritt

Das Bild liegt jetzt als NumPy-Array `ziffer_bild` mit Form `(28, 28)` vor – genau wie ein MNIST-Bild.

Im nächsten Schritt werden wir:
- Den **MNIST-Datensatz** laden (70.000 handgeschriebene Ziffern)
- Einen **Classifier** darauf trainieren (z.B. k-Nearest Neighbors oder ein neuronales Netz)
- Unsere selbst gemalte Ziffer vom Modell **erkennen** lassen